<a href="https://colab.research.google.com/github/chinwejoseph/JDA_NPOWER_CANADA/blob/main/Dash_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#!/usr/bin/env python
# coding: utf-8

# In[ ]:

!pip install dash
import dash
import more_itertools
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 46.0 MB/s eta 0:00:00


In [3]:
# Load the data using pandas
data = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/d51iMGfp_t0QpO30Lym-dw/automobile-sales.csv')

# Initialize the Dash app
app = dash.Dash(__name__)

In [5]:
# Set the title of the dashboard
#app.title = "Automobile Statistics Dashboard"
# Create the dropdown menu options
dropdown_options = [
    {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
    {'label': 'Recession Period Statistics', 'value': "Recession Period Statistics"}
]

In [7]:
# List of years
year_list = [i for i in range(1980, 2024, 1)]
year_list

[1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023]

In [8]:
# Create the layout of the app
app.layout = html.Div([])

In [14]:
# Add title to the dashboard
# Include style for title
# Add two dropdown menus
app.layout = html.Div([
    html.H1("Automobile Statistics Dashboard", style={'textAlign': 'center', 'color': '#503D36', 'font-size': 24}),
    html.Div([
        html.Label("Select Statistics:"),
        dcc.Dropdown(
            id='dropdown-statistics',
            options=dropdown_options,
            value='Yearly Statistics',
            placeholder='Select a statistic type'
        )
    ]),
    html.Div(dcc.Dropdown(
            id='select-year',
            options=[{'label': i, 'value': i} for i in year_list],
            value=year_list[0] # Default to the first year
        )),
    html.Div(id='output-container', className='chart-grid', style={'display': 'flex'})
])

In [16]:
#Add a division for output display
html.Div(id='output-display', className='output-area', style={})

Div(id='output-display', className='output-area', style={})

In [25]:
#Creating Callbacks
# Define the callback function to update the input container based on the selected statistics
@app.callback(
    Output(component_id='select-year', component_property='disabled'),
    Input(component_id='dropdown-statistics',component_property='value'))

def update_input_container(selected_statistics):
    if selected_statistics == 'Yearly Statistics':
        return False
    else:
        return True

#Callback for plotting
# Define the callback function to update the input container based on the selected statistics
@app.callback(
    Output(component_id='output-container', component_property='children'),
    [Input(component_id='dropdown-statistics', component_property='value'), Input(component_id='select-year', component_property='value')])

def update_output_container(selected_statistics, selected_year):
    if selected_statistics == 'Recession Period Statistics':
        # Filter the data for recession periods
        recession_data = data[data['Recession'] == 1]
        #Plot 1 Automobile sales fluctuate over Recession Period (year wise)
        # use groupby to create relevant data for plotting
        yearly_rec=recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        R_chart1 = dcc.Graph(
            figure=px.line(yearly_rec,
                x='Year',
                y='Automobile_Sales',
                title="Average Automobile Sales fluctuation over Recession Period"))

        # Plot 2 Calculate the average number of vehicles sold by vehicle type
        average_sales = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        R_chart2 = dcc.Graph(
            figure=px.bar(average_sales,
            x='Vehicle_Type',
            y='Automobile_Sales',
            title="Average Number of Vehicles Sold by Type during Recession"))

        # Plot 3 Pie chart for total expenditure share by vehicle type during recessions
        exp_rec = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        R_chart3 = dcc.Graph(
            figure=px.pie(exp_rec,
            values='Advertising_Expenditure',
            names='Vehicle_Type',
            title="Total Advertising Expenditure by Vehicle Type During Recession"))

        # Plot 4 bar chart for the effect of unemployment rate on vehicle type and sales
        unemp_data = recession_data.groupby(['Vehicle_Type', 'unemployment_rate'])['Automobile_Sales'].mean().reset_index()
        R_chart4 = dcc.Graph(figure=px.bar(unemp_data,
        x='unemployment_rate',
        y='Automobile_Sales',
        color='Vehicle_Type',
        labels={'unemployment_rate': 'Unemployment Rate', 'Automobile_Sales': 'Average Automobile Sales'},
        title='Effect of Unemployment Rate on Vehicle Type and Sales'))

        return html.Div([
             html.Div(className='chart-item', children=[html.Div(children=R_chart1),html.Div(children=R_chart2)],style={'display': 'flex'}),
             html.Div(className='chart-item', children=[html.Div(children=R_chart3), html.Div(children=R_chart4)],style={'display': 'flex'})
            ])
    elif selected_statistics == 'Yearly Statistics':
        # Filter data for the selected year
        yearly_data = data[data['Year'] == selected_year]

        # Plot 1: Total Automobile Sales by Vehicle Type in the selected year
        yas_data = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].sum().reset_index()
        Y_chart1 = dcc.Graph(figure=px.bar(yas_data,
                                            x='Vehicle_Type',
                                            y='Automobile_Sales',
                                            title=f'Automobile Sales by Vehicle Type in {selected_year}'))

        # Plot 2: Average Wholesale Cost by Vehicle Type in the selected year
        avg_wholesale_data = yearly_data.groupby('Vehicle_Type')['Wholesale_Cost'].mean().reset_index()
        Y_chart2 = dcc.Graph(figure=px.bar(avg_wholesale_data,
                                            x='Vehicle_Type',
                                            y='Wholesale_Cost',
                                            title=f'Average Wholesale Cost by Vehicle Type in {selected_year}'))

        # Plot 3: Total Advertising Expenditure by Vehicle Type in the selected year
        adv_exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        Y_chart3 = dcc.Graph(figure=px.pie(adv_exp_data,
                                            values='Advertising_Expenditure',
                                            names='Vehicle_Type',
                                            title=f'Total Advertising Expenditure by Vehicle Type in {selected_year}'))

        return html.Div([
            html.Div(Y_chart1, className='chart-item'),
            html.Div(Y_chart2, className='chart-item'),
            html.Div(Y_chart3, className='chart-item')
        ], style={'display': 'flex', 'flex-wrap': 'wrap'})
    else:
        return html.Div("Please select a statistics type and year.")

In [28]:
# Run the Dash app
if __name__ == '__main__':
    app.run(debug=True)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: on
